In [ ]:
# Install required libraries including the new trace decoders
!pip install -q transformers datasets accelerate bitsandbytes teich torchcodec

# Mount your 1TB Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.2/906.2 kB 57.5 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import logging
from huggingface_hub import snapshot_download
from datasets import load_dataset
import sys

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

MODELS_TO_DOWNLOAD = [
    "Qwen/Qwen2.5-7B-Instruct",
    "BAAI/bge-large-en-v1.5"
]

DATASET_QUEUE = {
    # 1. Curated Frontier Model Traces (Reasoning & Agentic Flow)
    "frontier_traces": [
        "Crownelius/Complete-FABLE.5-traces-2M",
        "armand0e/claude-fable-5-claude-code",
        "ansulev/claude_mythos_distilled_25k",
        "ox-ox/mythos-character-distillation",
        "11-47/claude_opus_4.8_max_thinking_5k_v2",
        "Quaxicron/claude-opus-4.8-pi-traces",
        "angrygiraffe/claude-opus-4.6-4.7-reasoning-8.7k",
        "TeichAI/lordx64-claude-opus-4.7-max-cleaned",
        "Jackrong/Claude-opus-4.7-TraceInversion-5000x",
        "AletheiaResearch/GPT-5.5-Codex",
        "armand0e/gpt-5.5-agent",
        "armand0e/gpt-5.5-chat",
        "hotdogs/uka-glm-5.2",
        "AletheiaResearch/GLM-5.2-Bench",
        "armand0e/qwen3.7-max-pi-traces",
        "tomaarsen/zelo-scores-10kx100-qwen3.6-27b",
        "zake7749/Qwen3.6-35B-A3B-Tool-Calling",
        "khazarai/qwen3.6-plus-high-reasoning-500x",
        "caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b",
        "armand0e/minimax-m3-claude-code-traces",
        "Infatoshi/kernelbench-mega-traces",
        "Roman1111111/gemini-3.1-pro-hard-high-reasoning",
        "PhysEdit/pawbench-gemini-expansion-20260619",
        "mfielding92/gemini-3.1-pro-2048-reasoning-1100x",
        "rajpurkar/squad",
        "google/boolq",
        "benchflow/skillsbench-leaderboard",
        "evaleval/EEE_datastore"
    ],

    # 2. Massive General Knowledge, Instruction Following, & Creative Core
    "general_knowledge": [
        "HuggingFaceFW/fineweb-edu",
        "HuggingFaceH4/ultrafeedback_clean",
        "technium/OpenHermes-2.5",
        "KingNish/reasoning-base-20k",
        "Salesforce/wikitext",
        "banned-historical-archives/banned-historical-archives",
        "allenai/c4",
        "stanfordnlp/imdb",
        "legacy-datasets/wikipedia",
        "bookcorpus/bookcorpus",
        "fse/paranmt-300",
        "Skylion007/openwebtext",
        "evaluate-metric/xnli",
        "liwu/MNBVC"
    ],

    # 3. Code Syntax & Language Grammar Rules (For the JEPA World Model)
    "code_mechanics": [
        "m-a-p/CodeFeedback-Filtered-Instruction",
        "deepmind/code_contests",
        "code-search-net/code_search_net",
        "bigcode/starcoder2-instruct",
        "iamtarun/python-execution-traces",
        "bigcode/the-stack",
        "bookcorpus/bookcorpus",
        "Salesforce/wikisql",
        "gaianet/learn-rust",
        "semeru/code-code-translation-java-csharp",
        "MehdiFe/csharp-instruction-Dataset",
        "microsoft/LCC_csharp",
        "AlgorithmicResearchGroup/arxiv_cplusplus_research_code",
        "FradSer/DeepSeek-R1-Distilled-Translate-en-zh_CN-39k",
        "FradSer/DeepSeek-R1-Distilled-Translate-en-zh_CN-39k-Alpaca-GPT4",
        "bh2821/LightNovel5000"
    ]
}

def preflight_download():
    logger.info("Starting Pre-flight Model and Dataset Downloading...")

    # Download models
    logger.info("=== Downloading Models ===")
    for model_id in MODELS_TO_DOWNLOAD:
        logger.info(f"Downloading/Verifying model: {model_id}")
        try:
            snapshot_download(repo_id=model_id, local_files_only=False)
            logger.info(f"Successfully cached model: {model_id}")
        except Exception as e:
            logger.error(f"Failed to download model {model_id}: {e}")
            sys.exit(1)

    # Download datasets
    logger.info("=== Downloading Datasets ===")
    for domain, datasets in DATASET_QUEUE.items():
        for dataset_id in datasets:
            logger.info(f"Downloading/Verifying dataset: {dataset_id} (streaming cache)")
            try:
                # We just load a dummy subset to force cache resolution if not present,
                # though streaming doesn't strictly "download" everything locally beforehand.
                # Just ensuring the dataset is accessible.
                load_args = {"path": dataset_id, "split": "train", "streaming": True}
                if dataset_id == "HuggingFaceFW/fineweb-edu":
                    load_args["name"] = "sample-10BT"
                try:
                    ds = load_dataset(**load_args)
                except Exception:
                    # Retry without train split
                    ds = load_dataset(dataset_id, streaming=True)

                # Fetch one item to test
                next(iter(ds))
                logger.info(f"Successfully verified dataset: {dataset_id}")
            except Exception as e:
                logger.error(f"Failed to verify dataset {dataset_id}: {e}")
                # We don't exit here to allow failure on specific obscure datasets without crashing the whole pipeline

    logger.info("Pre-flight download completed successfully. All accessible assets are cached.")

if __name__ == "__main__":
    preflight_download()


2026-06-26 19:23:17,260 - INFO - Starting Pre-flight Model and Dataset Downloading...
2026-06-26 19:23:17,261 - INFO - === Downloading Models ===
2026-06-26 19:23:17,264 - INFO - Downloading/Verifying model: Qwen/Qwen2.5-7B-Instruct
2026-06-26 19:23:17,520 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/revision/main "HTTP/1.1 200 OK"


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

2026-06-26 19:23:17,804 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/.gitattributes "HTTP/1.1 307 Temporary Redirect"
2026-06-26 19:23:17,813 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/.gitattributes "HTTP/1.1 200 OK"
2026-06-26 19:23:17,824 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/.gitattributes "HTTP/1.1 200 OK"
2026-06-26 19:23:17,828 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-26 19:23:17,835 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/README.md "HTTP/1.1 307 Temporary Redirect"
2026-0

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

2026-06-26 19:34:54,824 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/d4aa6901d3a41ba39fb536a557fa166f842b0e09/.gitattributes "HTTP/1.1 307 Temporary Redirect"
2026-06-26 19:34:54,831 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/.gitattributes "HTTP/1.1 200 OK"
2026-06-26 19:34:54,841 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/d4aa6901d3a41ba39fb536a557fa166f842b0e09/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-26 19:34:54,846 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/.gitattributes "HTTP/1.1 200 OK"
2026-06-26 19:34:54,847 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/README.md "HTTP/1.1 200 OK"
2026-06-26 19:34:54,864 - IN

...

2026-06-26 19:49:55,133 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceFW/fineweb-edu/revision/87f09149ef4734204d70ed1d046ddc9ca3f2b8f9 "HTTP/1.1 200 OK"
2026-06-26 19:49:55,134 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceFW/fineweb-edu/revision/87f09149ef4734204d70ed1d046ddc9ca3f2b8f9 "HTTP/1.1 200 OK"
2026-06-26 19:50:37,429 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/87f09149ef4734204d70ed1d046ddc9ca3f2b8f9/dataset_infos.json "HTTP/1.1 404 Not Found"
2026-06-26 19:50:37,665 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceFW/fineweb-edu/tree/87f09149ef4734204d70ed1d046ddc9ca3f2b8f9/sample%2F10BT?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-06-26 19:50:37,899 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/HuggingFaceFW/fineweb-edu/tree/87f09149ef4734204d70ed1d046ddc9ca3f2b8f9/sample?recursive=false&expand=false "HTTP/1.1 200 OK"
202

README.md:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

2026-06-26 19:50:42,058 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/KingNish/reasoning-base-20k/resolve/48f49880a1dfe2f8fd3f06e7df610f09fbd4d06a/reasoning-base-20k.py "HTTP/1.1 404 Not Found"
2026-06-26 19:50:42,729 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/KingNish/reasoning-base-20k/KingNish/reasoning-base-20k.py "HTTP/1.1 404 Not Found"
2026-06-26 19:50:43,003 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/KingNish/reasoning-base-20k/revision/48f49880a1dfe2f8fd3f06e7df610f09fbd4d06a "HTTP/1.1 200 OK"
2026-06-26 19:50:43,248 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/KingNish/reasoning-base-20k/resolve/48f49880a1dfe2f8fd3f06e7df610f09fbd4d06a/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-06-26 19:50:43,505 - INFO - HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=KingNish/reasoning-base-20k "HTTP/1.1 200 OK"
2026-06-26 19:50:43,772 - INFO - HTTP Request: GET ht

README.md:   0%|          | 0.00/2.93k [00:00<?, ?B/s]

2026-06-26 19:50:52,870 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/m-a-p/CodeFeedback-Filtered-Instruction/resolve/a08c213a9748c66c15d0225814be80a2e77adf4a/CodeFeedback-Filtered-Instruction.py "HTTP/1.1 404 Not Found"
2026-06-26 19:50:53,555 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/m-a-p/CodeFeedback-Filtered-Instruction/m-a-p/CodeFeedback-Filtered-Instruction.py "HTTP/1.1 404 Not Found"
2026-06-26 19:50:53,790 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/m-a-p/CodeFeedback-Filtered-Instruction/revision/a08c213a9748c66c15d0225814be80a2e77adf4a "HTTP/1.1 200 OK"
2026-06-26 19:50:54,028 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/m-a-p/CodeFeedback-Filtered-Instruction/resolve/a08c213a9748c66c15d0225814be80a2e77adf4a/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-06-26 19:50:54,289 - INFO - HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=m-a-p/CodeFeedback-Filte

In [ ]:
import logging
import sys

# 1. Force stdout/stderr to be unbuffered at the system level
os.environ['PYTHONUNBUFFERED'] = '1'

# 2. Reset the root logger entirely
root = logging.getLogger()
root.handlers = [] # Clear existing handlers that might be suppressing output

# 3. Add a fresh, unbuffered handler
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
root.addHandler(handler)
root.setLevel(logging.INFO)

# 4. Final safety flush
import functools
print = functools.partial(print, flush=True)

import os # Ensure os is imported

import os
import re
import time
import logging
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Change this:
# logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# To this (adding a StreamHandler with flush):
import sys
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
# Explicitly force the handler to flush
for handler in logging.root.handlers:
    handler.flush = sys.stdout.flush

DATASET_QUEUE = {
    # 1. Curated Frontier Model Traces (Reasoning, Alignment & Agentic Flow)
    "frontier_traces": [
        "Crownelius/Complete-FABLE.5-traces-2M",
        "ansulev/claude_mythos_distilled_25k",
        "ox-ox/mythos-character-distillation",
        "11-47/claude_opus_4.8_max_thinking_5k_v2",
        "Quaxicron/claude-opus-4.8-pi-traces",
        "angrygiraffe/claude-opus-4.6-4.7-reasoning-8.7k",
        "TeichAI/lordx64-claude-opus-4.7-max-cleaned",
        "Jackrong/Claude-opus-4.7-TraceInversion-5000x",
        "AletheiaResearch/GPT-5.5-Codex",
        "armand0e/gpt-5.5-agent",
        "armand0e/gpt-5.5-chat",
        "hotdogs/uka-glm-5.2",
        "AletheiaResearch/GLM-5.2-Bench",
        "armand0e/qwen3.7-max-pi-traces",
        "tomaarsen/zelo-scores-10kx100-qwen3.6-27b",
        "zake7749/Qwen3.6-35B-A3B-Tool-Calling",
        "khazarai/qwen3.6-plus-high-reasoning-500x",
        "caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b",
        "armand0e/minimax-m3-claude-code-traces",
        "Infatoshi/kernelbench-mega-traces",
        "Roman1111111/gemini-3.1-pro-hard-high-reasoning",
        "PhysEdit/pawbench-gemini-expansion-20260619",
        "mfielding92/gemini-3.1-pro-2048-reasoning-1100x",
        "benchflow/skillsbench-leaderboard",
        "evaleval/EEE_datastore",
        # Parallel Translation Alignments belong here!
        "FradSer/DeepSeek-R1-Distilled-Translate-en-zh_CN-39k",
        "FradSer/DeepSeek-R1-Distilled-Translate-en-zh_CN-39k-Alpaca-GPT4",
        "bh2821/LightNovel5000"
    ],

    # 2. Massive General Knowledge, Instruction Following, & Creative Core
    "general_knowledge": [
        "HuggingFaceFW/fineweb-edu",
        "HuggingFaceH4/ultrafeedback_clean",
        "teknium/OpenHermes-2.5",
        "teknium/openhermes",
        "KingNish/reasoning-base-20k",
        "Salesforce/wikitext",
        "banned-historical-archives/banned-historical-archives",
        "allenai/c4",
        "stanfordnlp/imdb",
        "legacy-datasets/wikipedia",
        "bookcorpus/bookcorpus",
        "fse/paranmt-300",
        "Skylion007/openwebtext",
        "evaluate-metric/xnli",
        "liwu/MNBVC",
        "wdndev/webnovel-chinese",
        # Moved basic comprehension datasets here
        "rajpurkar/squad",
        "google/boolq"
    ],

    # 3. Code Syntax & Language Grammar Rules (For the JEPA World Model)
    "code_mechanics": [
        "m-a-p/CodeFeedback-Filtered-Instruction",
        "deepmind/code_contests",
        "code-search-net/code_search_net",
        "bigcode/starcoder2-instruct",
        "iamtarun/python-execution-traces",
        "bigcode/the-stack",
        "Salesforce/wikisql",
        "gaianet/learn-rust",
        "semeru/code-code-translation-java-csharp",
        "MehdiFe/csharp-instruction-Dataset",
        "microsoft/LCC_csharp",
        "AlgorithmicResearchGroup/arxiv_cplusplus_research_code"
    ]
}

def stringify_complex(obj):
    if isinstance(obj, str):
        return obj
    if isinstance(obj, (list, dict)):
        try:
            return json.dumps(obj, ensure_ascii=False)
        except Exception:
            return str(obj)
    return str(obj)

def find_key_recursive(data, target_key):
    if isinstance(data, dict):
        if target_key in data:
            return data[target_key]
        for key, value in data.items():
            result = find_key_recursive(value, target_key)
            if result is not None:
                return result
    elif isinstance(data, list):
        for item in data:
            result = find_key_recursive(item, target_key)
            if result is not None:
                return result
    return None

def extract_all_strings(data):
    strings = []
    if isinstance(data, str):
        strings.append(data)
    elif isinstance(data, dict):
        for val in data.values():
            strings.extend(extract_all_strings(val))
    elif isinstance(data, list):
        for item in data:
            strings.extend(extract_all_strings(item))
    return strings

def extract_qa_pair(row, dataset_name):
    prompt, response = None, None
    try:
        messages = find_key_recursive(row, "messages")
        if not messages:
            messages = find_key_recursive(row, "conversations")

        if messages and isinstance(messages, list) and len(messages) >= 2:
            prompt_parts = []
            response_content = None

            for msg in messages:
                if not isinstance(msg, dict): continue
                role = str(msg.get("role", msg.get("from", ""))).lower()
                content = stringify_complex(msg.get("content", msg.get("value", "")))

                if role in ["system", "user", "human", "prompter"] and not prompt_parts:
                    prompt_parts.append(f"{role}: {content}")
                    break

            for msg in reversed(messages):
                if not isinstance(msg, dict): continue
                role = str(msg.get("role", msg.get("from", ""))).lower()
                content = stringify_complex(msg.get("content", msg.get("value", "")))

                if role in ["assistant", "bot", "model", "gpt"]:
                    response_content = content
                    break

            if prompt_parts and response_content:
                prompt = "\n".join(prompt_parts)
                return prompt, response_content

        if "instruction" in row and "output" in row:
            prompt = stringify_complex(row["instruction"])
            if "input" in row and row["input"]:
                prompt += f"\n\nContext: {stringify_complex(row['input'])}"
            response = stringify_complex(row["output"])
            return prompt, response

        if "prompt" in row and "response" in row:
            return stringify_complex(row["prompt"]), stringify_complex(row["response"])
        if "question" in row and "answer" in row:
            return stringify_complex(row["question"]), stringify_complex(row["answer"])

        if "prompt" in row and "completion" in row:
            return stringify_complex(row["prompt"]), stringify_complex(row["completion"])

        # Add this rule inside your extract_qa_pair function in extract_frontier_data.py
        if "zh" in row and "en" in row:
            return stringify_complex(row["zh"]), stringify_complex(row["en"])

        if "text" in row:
            text = stringify_complex(row["text"])
            if len(text) > 10:
                mid = len(text) // 2
                prompt = text[:mid]
                response = text[mid:]
                return prompt, response

        all_strings = extract_all_strings(row)
        if all_strings:
            longest_string = max(all_strings, key=len)
            if len(longest_string) > 10:
                mid = len(longest_string) // 2
                prompt = longest_string[:mid]
                response = longest_string[mid:]
                return prompt, response

    except Exception as e:
        logging.debug(f"Failed to parse row: {e}")

    return None, None

def process_datasets(save_dir="/content/drive/MyDrive/JEPA_Shards", chunk_size=10000):
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    # Note: Swapped to CUDA for Colab T4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logging.info(f"Using device: {device}")

    model_id = "Qwen/Qwen2.5-7B-Instruct"
    encoder_id = "BAAI/bge-m3"

    logging.info(f"Loading tokenizer: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    logging.info(f"Loading encoder: {encoder_id}")
    encoder_tokenizer = AutoTokenizer.from_pretrained(encoder_id)
    encoder_model = AutoModel.from_pretrained(encoder_id).to(device)
    encoder_model.eval()

    start_time = time.time()
    for domain, datasets in DATASET_QUEUE.items():
        for dataset_name in datasets:
            logging.info(f"Starting ingestion for: {dataset_name} in domain: {domain}")
            safe_name = dataset_name.replace("/", "_")
            buffer = []
            chunk_id = 0

            load_args = {"path": dataset_name, "split": "train", "streaming": True}
            if dataset_name == "HuggingFaceFW/fineweb-edu":
                load_args["name"] = "sample-100BT"
            elif dataset_name == "liwu/MNBVC":
                load_args["name"] = "web_novel"

            try:
                ds = load_dataset(**load_args)
            except Exception as e:
                logging.warning(f"Train split not found for {dataset_name}, falling back to default: {e}")
                try:
                    ds = load_dataset(dataset_name, streaming=True)
                except Exception as e2:
                    logging.error(f"Completely failed to load {dataset_name}: {e2}")
                    continue

            # Scale limits for Colab (FineWeb maxed to 10M rows)
            if domain == "general_knowledge":
                max_rows = 20_000_000
            elif dataset_name == "wdndev/webnovel-chinese":
                max_rows = 5_000_000   # Solid base for Chinese prose structures
            elif domain == "frontier_traces":
                max_rows = 500_000
            else:
                max_rows = 1_000_000

            rows_processed = 0

            # Native Skip Resume Logic
            existing_shards = list(save_path.glob(f"{domain}_{safe_name}_*.pt"))
            if existing_shards:
                chunk_ids = []
                for shard in existing_shards:
                    match = re.search(r"_(\d+)\.pt$", shard.name)
                    if match:
                        chunk_ids.append(int(match.group(1)))

                if chunk_ids:
                    highest_chunk_id = max(chunk_ids)
                    chunk_id = highest_chunk_id + 1
                    rows_to_skip = chunk_id * chunk_size

                    if rows_to_skip > 0:
                        logging.info(f"Resuming {dataset_name}: Fast-forwarding {rows_to_skip} rows...")
                        ds = ds.skip(rows_to_skip)
                        rows_processed = rows_to_skip

            try:
                ds_iterator = iter(ds)
                while True:
                    if rows_processed >= max_rows:
                        break

                    try:
                        row = next(ds_iterator)
                    except StopIteration:
                        break
                    except Exception as e:
                        logging.error(f"Dataset {dataset_name} iterator crashed: {e}. Moving to next dataset.")
                        break

                    try:
                        prompt, response = extract_qa_pair(row, dataset_name)
                        if not prompt or not response:
                            continue

                        input_tokens = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)["input_ids"].squeeze(0)
                        qwen_tokens = tokenizer(response, return_tensors="pt", truncation=True, max_length=2048)["input_ids"].squeeze(0)

                        if input_tokens.numel() == 0 or qwen_tokens.numel() == 0:
                            continue

                        enc_inputs = encoder_tokenizer(response, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)

                        with torch.no_grad():
                            enc_outputs = encoder_model(**enc_inputs)
                            cls_embedding = enc_outputs.last_hidden_state[:, 0, :]
                            target_concept = F.normalize(cls_embedding, p=2, dim=1).squeeze(0).cpu()

                        buffer.append({
                            "input_tokens": input_tokens.cpu(),
                            "qwen_tokens": qwen_tokens.cpu(),
                            "target_concept": target_concept
                        })
                        rows_processed += 1

                        if len(buffer) >= chunk_size:
                            shard_path = save_path / f"{domain}_{safe_name}_{chunk_id}.pt"
                            torch.save(buffer, shard_path)
                            elapsed = time.time() - start_time
                            throughput = len(buffer) / elapsed if elapsed > 0 else 0
                            logging.info(f"Saved {shard_path} with {len(buffer)} items. Speed: {throughput:.2f} samples/sec")
                            start_time = time.time()
                            buffer = []
                            chunk_id += 1

                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()

                    except Exception as e:
                        logging.warning(f"Error processing row in {dataset_name}: {e}")
                        continue
            except Exception as e:
                logging.error(f"Critical error during {dataset_name} processing: {e}")

            if buffer:
                shard_path = save_path / f"{domain}_{safe_name}_{chunk_id}.pt"
                torch.save(buffer, shard_path)
                elapsed = time.time() - start_time
                throughput = len(buffer) / elapsed if elapsed > 0 else 0
                logging.info(f"Saved final {shard_path} with {len(buffer)} items. Speed: {throughput:.2f} samples/sec")
                start_time = time.time()

            logging.info(f"Finished {dataset_name}. Total processed pairs: {rows_processed}")

if __name__ == "__main__":
    process_datasets(save_dir="/content/drive/MyDrive/JEPA_Shards", chunk_size=10000)

2026-06-27 01:19:06,651 - INFO - Using device: cuda
2026-06-27 01:19:06,654 - INFO - Loading tokenizer: Qwen/Qwen2.5-7B-Instruct
2026-06-27 01:19:06,923 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-27 01:19:06,932 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
2026-06-27 01:19:07,242 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-27 01:19:07,255 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-27 01:19:07,503 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/tree/main/additional_chat_templ

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-06-27 01:19:18,151 - INFO - Starting ingestion for: Crownelius/Complete-FABLE.5-traces-2M in domain: frontier_traces
2026-06-27 01:19:18,426 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/Crownelius/Complete-FABLE.5-traces-2M/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-27 01:19:18,451 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Crownelius/Complete-FABLE.5-traces-2M/19a5b7863e10eec6838cf531bd20d24d2ec1106e/README.md "HTTP/1.1 200 OK"
2026-06-27 01:19:18,705 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/Crownelius/Complete-FABLE.5-traces-2M/resolve/19a5b7863e10eec6838cf531bd20d24d2ec1106e/Complete-FABLE.5-traces-2M.py "HTTP/1.1 404 Not Found"
2026-06-27 01:19:19,458 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Crownelius/Complete-FABLE.5-traces-2M/Crownelius/Complete-FABLE.5-traces-2M.py "HTTP/1.1 404 Not Found"
2026-06-27 01:19:19,733 - INFO - HTTP Req

KeyboardInterrupt: 